Pull complete OSDR file listings for both studies

In [1]:

import urllib.request, json

def get_files(osd_id):
    url = f"https://osdr.nasa.gov/osdr/data/osd/files/{osd_id}/?all_files=true&page_size=1000"
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        data = json.load(r)
    return data

for osd in [767, 766]:
    d = get_files(osd)
    studies = d.get("studies", {})
    key = f"OSD-{osd}"
    info = studies.get(key, {})
    files = info.get("study_files", [])
    print(f"\n===== {key}  (total files: {info.get('file_count')}, returned: {len(files)}) =====")
    # Group by category then subcategory, exclude raw fastq
    from collections import defaultdict
    by_cat = defaultdict(list)
    for f in files:
        by_cat[(f.get("category",""), f.get("subcategory",""))].append(f)
    for (cat, sub), fs in sorted(by_cat.items()):
        non_fastq = [f for f in fs if not f["file_name"].endswith(".fastq.gz")]
        if non_fastq:
            print(f"\n  [{cat}] / [{sub}]  ({len(non_fastq)} non-fastq files)")
            for f in non_fastq[:15]:
                sz = f.get("file_size",0)
                print(f"     {f['file_name']}  ({sz/1e6:.2f} MB)")
            if len(non_fastq) > 15:
                print(f"     ... +{len(non_fastq)-15} more")
    # count fastq
    n_fastq = sum(1 for f in files if f["file_name"].endswith(".fastq.gz"))
    print(f"\n  Raw FASTQ files: {n_fastq}")



===== OSD-767  (total files: 638, returned: 638) =====

  [GeneLab Processed RNA-Seq Files] / [Aligned Sequence Data]  (182 non-fastq files)
     GLDS-709_rna_seq_VEG-05-Gnd-SN12-Leaf-Blue_L008_GLbulkRNAseq_SJ.out.tab  (3.86 MB)
     GLDS-709_rna_seq_VEG-05-Gnd-SN12-Leaf-Blue_L008_GLbulkRNAseq_Aligned.toTranscriptome.out.bam  (622.61 MB)
     GLDS-709_rna_seq_VEG-05-Gnd-SN12-Leaf-Blue_L008_GLbulkRNAseq_Aligned.sortedByCoord_sorted.out.bam.bai  (0.69 MB)
     GLDS-709_rna_seq_VEG-05-Gnd-SN12-Leaf-Blue_L008_GLbulkRNAseq_Log.final.out  (0.00 MB)
     GLDS-709_rna_seq_VEG-05-Gnd-SN12-Leaf-Blue_L008_GLbulkRNAseq_Aligned.sortedByCoord_sorted.out.bam  (545.60 MB)
     GLDS-709_rna_seq_VEG-05-Gnd-SN11-Leaf-Blue_L008_GLbulkRNAseq_Log.final.out  (0.00 MB)
     GLDS-709_rna_seq_VEG-05-Gnd-SN10-Leaf-Blue_L008_GLbulkRNAseq_Log.final.out  (0.00 MB)
     GLDS-709_rna_seq_VEG-05-Gnd-SN10-Adv-Root-Blue_L008_GLbulkRNAseq_Log.final.out  (0.00 MB)
     GLDS-709_rna_seq_VEG-05-Gnd-SN09-Leaf-Blue_L008_GLbu

Parse and compare sample structure across both studies

In [3]:

import urllib.request, json, re
from collections import defaultdict

def get_meta(osd_id):
    url = f"https://osdr.nasa.gov/osdr/data/osd/meta/{osd_id}"
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

def get_files(osd_id):
    url = f"https://osdr.nasa.gov/osdr/data/osd/files/{osd_id}/?all_files=true&page_size=1000"
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

# ---- RNA-seq samples (OSD-767) ----
m767 = get_meta(767)
study = m767["studies"]["studies"][0]
rna_samples = []
for s in study["assays"][0]["materials"]["samples"]:
    name = s["name"]
    rna_samples.append(name)

print(f"OSD-767 RNA-seq: {len(rna_samples)} samples")
# Parse: VEG-05-Flt-SN01-Adv-Root-Red_L008 or VEG-05-Gnd-SN01-Leaf-Red_L008
rna_parsed = []
for n in rna_samples:
    # remove _L008 suffix
    base = n.replace("_L008","")
    m = re.match(r'VEG-05-(Flt|Gnd)-SN(\d+)-(Adv-Root|Leaf)-(Red|Blue)', base)
    if m:
        rna_parsed.append({
            "raw": n, "flight": m.group(1), "plant": f"SN{m.group(2)}",
            "tissue": m.group(3), "light": m.group(4)
        })
    else:
        print(f"  UNPARSED: {n}")

print(f"  Parsed: {len(rna_parsed)}")
# Summary
from collections import Counter
print("  Flight/Ground:", Counter(r["flight"] for r in rna_parsed))
print("  Tissue:", Counter(r["tissue"] for r in rna_parsed))
print("  Light:", Counter(r["light"] for r in rna_parsed))
print("  Plants:", sorted(set(r["plant"] for r in rna_parsed)))

# ---- Microbiome samples (OSD-766) ----
f766 = get_files(766)
files = f766["studies"]["OSD-766"]["study_files"]
fastq = [f for f in files if f["file_name"].endswith(".fastq.gz") and "_R1_" in f["file_name"]]
print(f"\nOSD-766 Microbiome: {len(fastq)} R1 fastq files (samples)")

# Parse names like: GLDS-672_Amplicon_16S-1-VEG-05-F-tomato-SN-01-wick_S1_L001_R1_raw.fastq.gz
# or GLDS-672_Amplicon_16S-109-VEG-05F-SN01-leaf-tom-red-rich_S107_L001_R1_raw.fastq.gz
# or GLDS-672_Amplicon_16S-127-VEG-05-G-SN-01-leaf-tom-red-rich_S125_L001_R1_raw.fastq.gz
amp_types = Counter()
micro_parsed = []
for f in fastq:
    fn = f["file_name"]
    # Determine amplicon type
    amp = "16S" if "_16S-" in fn else ("ITS" if "_ITS-" in fn else "?")
    amp_types[amp] += 1
    # Extract the descriptive part between amplicon number and _Sxx
    m = re.search(r'Amplicon_(16S|ITS)-\d+-(.+?)_S\d+_L001', fn)
    if m:
        desc = m.group(2)
        micro_parsed.append({"raw": fn, "amplicon": amp, "desc": desc})

print(f"  Amplicon types: {dict(amp_types)}")
print(f"  Parsed: {len(micro_parsed)}")

# Parse the descriptive parts - they have varied formats
# Let's look at unique patterns
descs = [p["desc"] for p in micro_parsed]
# Categorize by tissue
tissue_patterns = {
    "wick": r'wick',
    "root": r'(?<!Adv)root(?!s)',  # root but not AdvRoot
    "AdvRoot": r'AdvRoot',
    "soil": r'soil',
    "leaf": r'leaf',
    "water": r'water',
}
tissue_counts = Counter()
flight_counts = Counter()
light_counts = Counter()
plant_set = set()
for d in descs:
    t = "unknown"
    for tname, pat in tissue_patterns.items():
        if re.search(pat, d, re.IGNORECASE):
            t = tname
            break
    tissue_counts[t] += 1
    # flight
    if re.search(r'VEG-05-F-|VEG-05F-', d):
        flight_counts["Flight"] += 1
    elif re.search(r'VEG-05-G-', d):
        flight_counts["Ground"] += 1
    # light
    if re.search(r'red-rich|red', d, re.IGNORECASE) and 'blue' not in d.lower():
        light_counts["Red"] += 1
    elif re.search(r'blue-rich|blue', d, re.IGNORECASE):
        light_counts["Blue"] += 1
    # plant
    pm = re.search(r'SN-?(\d+)', d)
    if pm:
        plant_set.add(f"SN{pm.group(1)}")

print(f"  Tissues: {dict(tissue_counts)}")
print(f"  Flight/Ground: {dict(flight_counts)}")
print(f"  Light: {dict(light_counts)}")
print(f"  Plants: {sorted(plant_set)}")

# Show some example descriptions per tissue
print("\n  Example descriptions per tissue:")
for t in ["leaf","root","AdvRoot","wick","soil","water"]:
    exs = [d for d in descs if re.search(tissue_patterns.get(t,''), d, re.IGNORECASE)][:3]
    if exs:
        print(f"    {t}: {exs}")


---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
Cell In[1], line 18
     14         return json.load(r)
     15 
     16 # ---- RNA-seq samples (OSD-767) ----
     17 m767 = get_meta(767)
---> 18 study = m767["studies"]["studies"][0]
     19 rna_samples = []
     20 for s in study["assays"][0]["materials"]["samples"]:
     21     name = s["name"]

KeyError: 'studies'

Inspect OSDR metadata JSON structure

In [5]:

import urllib.request, json, re
from collections import Counter

def get_meta(osd_id):
    url = f"https://osdr.nasa.gov/osdr/data/osd/meta/{osd_id}"
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

m767 = get_meta(767)
print("Top-level keys:", list(m767.keys()))
# The structure may be nested differently
k = list(m767.keys())[0]
print("First key:", k, "->", list(m767[k].keys()) if isinstance(m767[k], dict) else type(m767[k]))


Top-level keys: ['hits', 'input', 'study', 'success']
First key: hits -> <class 'int'>


Inspect study metadata structure

In [7]:

import urllib.request, json, re
from collections import Counter

def get_meta(osd_id):
    url = f"https://osdr.nasa.gov/osdr/data/osd/meta/{osd_id}"
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

m767 = get_meta(767)
s = m767["study"]
print("study keys:", list(s.keys()) if isinstance(s, dict) else type(s))
if isinstance(s, dict):
    for k in list(s.keys())[:10]:
        v = s[k]
        print(f"  {k}: {type(v).__name__}", (str(v)[:120] if not isinstance(v,(dict,list)) else f"len={len(v)}"))


study keys: ['OSD-767']
  OSD-767: dict len=6


Inspect OSD-767 study fields

In [9]:

import urllib.request, json, re
from collections import Counter

def get_meta(osd_id):
    url = f"https://osdr.nasa.gov/osdr/data/osd/meta/{osd_id}"
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

m767 = get_meta(767)
s = m767["study"]["OSD-767"]
print("OSD-767 keys:", list(s.keys()))
for k in s.keys():
    v = s[k]
    print(f"  {k}: {type(v).__name__}", (str(v)[:200] if not isinstance(v,(dict,list)) else f"len={len(v)}"))


OSD-767 keys: ['comments', 'ontologySourceReferences', 'people', 'publications', 'studies', 'version']
  comments: list len=0
  ontologySourceReferences: list len=5
  people: list len=0
  publications: list len=0
  studies: list len=1
  version: int 2


Parse sample structure across both studies

In [11]:

import urllib.request, json, re
from collections import Counter

def get_meta(osd_id):
    url = f"https://osdr.nasa.gov/osdr/data/osd/meta/{osd_id}"
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

def get_files(osd_id):
    url = f"https://osdr.nasa.gov/osdr/data/osd/files/{osd_id}/?all_files=true&page_size=1000"
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

# ---- RNA-seq samples (OSD-767) ----
m767 = get_meta(767)
study = m767["study"]["OSD-767"]["studies"][0]
print("Study title:", study.get("title",""))
print("Study description:", str(study.get("publicReleaseDate",""))[:200])
# factors
print("\nFactor categories:")
for fv in study["assays"][0]["materials"]["samples"][0].get("factorValues",[]):
    cat = fv.get("category",{})
    print(f"  factor id: {cat.get('@id','')}")
# Get sample names
rna_samples = [s["name"] for s in study["assays"][0]["materials"]["samples"]]
print(f"\nOSD-767 RNA-seq: {len(rna_samples)} samples")

# Parse
rna_parsed = []
for n in rna_samples:
    base = n.replace("_L008","")
    m = re.match(r'VEG-05-(Flt|Gnd)-SN(\d+)-(Adv-Root|Leaf)-(Red|Blue)', base)
    if m:
        rna_parsed.append({"raw": n, "flight": m.group(1), "plant": f"SN{m.group(2)}",
                           "tissue": m.group(3), "light": m.group(4)})
    else:
        print(f"  UNPARSED: {n}")

print(f"  Parsed: {len(rna_parsed)}")
print("  Flight/Ground:", dict(Counter(r["flight"] for r in rna_parsed)))
print("  Tissue:", dict(Counter(r["tissue"] for r in rna_parsed)))
print("  Light:", dict(Counter(r["light"] for r in rna_parsed)))
print("  Plants:", sorted(set(r["plant"] for r in rna_parsed)))

# ---- Microbiome samples (OSD-766) ----
f766 = get_files(766)
files = f766["studies"]["OSD-766"]["study_files"]
fastq_r1 = [f for f in files if f["file_name"].endswith(".fastq.gz") and "_R1_" in f["file_name"]]
print(f"\nOSD-766 Microbiome: {len(fastq_r1)} R1 fastq (samples)")

amp_types = Counter()
micro_parsed = []
for f in fastq_r1:
    fn = f["file_name"]
    amp = "16S" if "_16S-" in fn else ("ITS" if "_ITS-" in fn else "?")
    amp_types[amp] += 1
    m = re.search(r'Amplicon_(16S|ITS)-\d+-(.+?)_S\d+_L001', fn)
    if m:
        micro_parsed.append({"raw": fn, "amplicon": amp, "desc": m.group(2)})

print(f"  Amplicon types: {dict(amp_types)}")
print(f"  Parsed: {len(micro_parsed)}")

# Parse tissue/flight/light/plant from descriptions
tissue_counts = Counter()
flight_counts = Counter()
light_counts = Counter()
plant_set = set()
for p in micro_parsed:
    d = p["desc"]
    # tissue
    if re.search(r'AdvRoot', d, re.I): t="AdvRoot"
    elif re.search(r'(?<!\w)root(?!s)', d, re.I): t="root"
    elif re.search(r'leaf', d, re.I): t="leaf"
    elif re.search(r'wick', d, re.I): t="wick"
    elif re.search(r'soil', d, re.I): t="soil"
    elif re.search(r'water', d, re.I): t="water"
    else: t="unknown"
    tissue_counts[t]+=1
    # flight
    if re.search(r'VEG-05-F-|VEG-05F-', d): flight_counts["Flight"]+=1
    elif re.search(r'VEG-05-G-', d): flight_counts["Ground"]+=1
    else: flight_counts["unknown"]+=1
    # light
    if re.search(r'blue', d, re.I): light_counts["Blue"]+=1
    elif re.search(r'red', d, re.I): light_counts["Red"]+=1
    else: light_counts["unknown"]+=1
    # plant
    pm = re.search(r'SN-?(\d+)', d)
    if pm: plant_set.add(f"SN{pm.group(1)}")

print(f"  Tissues: {dict(tissue_counts)}")
print(f"  Flight/Ground: {dict(flight_counts)}")
print(f"  Light: {dict(light_counts)}")
print(f"  Plants: {sorted(plant_set)}")

# Show example descriptions
print("\n  Example descriptions:")
for t in ["leaf","root","AdvRoot","wick","soil","water"]:
    exs = [p["desc"] for p in micro_parsed if t in p["desc"].lower()][:2]
    if exs: print(f"    {t}: {exs}")


Study title: Stress and light spectral quality influence the transcriptome of a tomato crop on the International Space Station
Study description: 09-Jun-2026

Factor categories:
  factor id: WJFkWznuAdC0XbvXC8jUodXStSE2IXesgcsoQoDm
  factor id: rrAHixexWAdqOKNaj15Ezk66kWZSMcprcBRZfg1k
  factor id: LajVshKKMiElHRpGA9Rjdp82m03F4bQ5UDeiHawX

OSD-767 RNA-seq: 36 samples
  Parsed: 36
  Flight/Ground: {'Flt': 18, 'Gnd': 18}
  Tissue: {'Adv-Root': 15, 'Leaf': 21}
  Light: {'Red': 19, 'Blue': 17}
  Plants: ['SN01', 'SN010', 'SN012', 'SN02', 'SN03', 'SN04', 'SN05', 'SN06', 'SN07', 'SN08', 'SN09', 'SN10', 'SN11', 'SN12']

OSD-766 Microbiome: 283 R1 fastq (samples)
  Amplicon types: {'16S': 142, 'ITS': 141}
  Parsed: 283
  Tissues: {'soil': 47, 'root': 20, 'wick': 16, 'unknown': 120, 'water': 8, 'AdvRoot': 30, 'leaf': 42}
  Flight/Ground: {'Ground': 172, 'Flight': 111}
  Light: {'unknown': 175, 'Red': 50, 'Blue': 58}
  Plants: ['SN003', 'SN006', 'SN01', 'SN010', 'SN012', 'SN02', 'SN03', 'SN04', '

Get study descriptions and resolve unknown microbiome tissues

In [13]:

import urllib.request, json, re
from collections import Counter

def get_meta(osd_id):
    url = f"https://osdr.nasa.gov/osdr/data/osd/meta/{osd_id}"
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

def get_files(osd_id):
    url = f"https://osdr.nasa.gov/osdr/data/osd/files/{osd_id}/?all_files=true&page_size=1000"
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

# Study descriptions
for osd in [766, 767]:
    m = get_meta(osd)
    s = m["study"][f"OSD-{osd}"]["studies"][0]
    print(f"\n=== OSD-{osd}: {s.get('title','')} ===")
    # Find description in comments or study description
    desc = s.get("description", "")
    if not desc:
        for c in s.get("comments", []):
            if "description" in c.get("name","").lower() or "Description" in c.get("name",""):
                desc = c.get("value","")
    print(f"Description: {str(desc)[:600]}")
    # Look for study comments
    for c in s.get("comments", [])[:8]:
        print(f"  Comment: {c.get('name','')}: {str(c.get('value',''))[:150]}")

# Now resolve unknown tissues in microbiome
f766 = get_files(766)
files = f766["studies"]["OSD-766"]["study_files"]
fastq_r1 = [f for f in files if f["file_name"].endswith(".fastq.gz") and "_R1_" in f["file_name"]]

unknown_descs = []
for f in fastq_r1:
    fn = f["file_name"]
    m = re.search(r'Amplicon_(16S|ITS)-\d+-(.+?)_S\d+_L001', fn)
    if m:
        d = m.group(2)
        if not any(re.search(p, d, re.I) for p in [r'AdvRoot', r'(?<!\w)root(?!s)', r'leaf', r'wick', r'soil', r'water']):
            unknown_descs.append((m.group(1), d))

print(f"\n\nUnknown tissue descriptions ({len(unknown_descs)}):")
for amp, d in sorted(set(unknown_descs)):
    print(f"  [{amp}] {d}")



=== OSD-766: The microbial communities of a tomato crop grown in Veggie under different lighting regimes on the International Space Station ===
Description: The VEG-05 experiment was conducted on ISS between 12/14/2022-3/24/2023 (101 days) to investigate tomato-growth responses and plant-microbiome differences due to two lighting recipes:  90%R:10%B as the “red-rich” treatment and 50%R:50%B as the “blue-rich” treatment.  A ground control was performed in controlled environment chambers at Kennedy Space Center providing. After a 48-hour delay, CO2 concentrations, relative humidity and temperature from data downlinked on the ISS were configured to run in the ground control growth chamber.  To capture humidity and temperature inside of the Veggie uni
  Comment: Acknowledgments: 
  Comment: DOI: 10.26030/ywjn-5e23
  Comment: Data Source Accession: 
  Comment: Data Source Link: 
  Comment: Experiment Platform: Vegetable Production System (Veggie)
  Comment: Flight Program: International Sp

Download RNA-seq runsheet to check reference genome

In [15]:

import urllib.request, json, re
from collections import Counter

def get_files(osd_id):
    url = f"https://osdr.nasa.gov/osdr/data/osd/files/{osd_id}/?all_files=true&page_size=1000"
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

# Check RNA-seq runsheet for reference genome info
f767 = get_files(767)
files767 = f767["studies"]["OSD-767"]["study_files"]
runsheet = [f for f in files767 if "runsheet" in f["file_name"].lower()]
print("Runsheet file:", runsheet)
# Download runsheet
if runsheet:
    url = "https://osdr.nasa.gov" + runsheet[0]["remote_url"]
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        content = r.read().decode()
    # Print first 30 lines to see reference info
    lines = content.split("\n")
    for i, line in enumerate(lines[:40]):
        print(f"  {i}: {line[:200]}")


Runsheet file: [{'category': 'GeneLab Processed RNA-Seq Files', 'date_created': 1781824028.177, 'date_updated': 1781824028.177, 'file_name': 'GLDS-709_rna_seq_bulkRNASeq_v2_runsheet.csv', 'file_size': 17015, 'organization': 'genelab', 'remote_url': '/geode-py/ws/studies/OSD-767/download?source=datamanager&file=GLDS-709_rna_seq_bulkRNASeq_v2_runsheet.csv', 'restricted': False, 'subcategory': 'Processing Info', 'subdirectory': '', 'visible': True}]
  0: Sample Name,has_ERCC,organism,paired_end,read1_path,read2_path,Factor Value[Spaceflight],Factor Value[light regimen],Factor Value[Organism Part],Source Name,Has Tech Reps,Original Sample Name
  1: VEG-05-Flt-SN05-Leaf-Red_L008,False,Solanum lycopersicum,True,https://osdr.nasa.gov/geode-py/ws/studies/OSD-767/download?source=datamanager&file=GLDS-709_rna-seq_VEG-05-Flt-SN05-Leaf-Red_L008_R1_raw.f
  2: VEG-05-Gnd-SN07-Adv-Root-Blue_L008,False,Solanum lycopersicum,True,https://osdr.nasa.gov/geode-py/ws/studies/OSD-767/download?source=datamana

Check RNA-seq software versions and reference genome

In [17]:

import urllib.request, json, re
from collections import Counter

def get_files(osd_id):
    url = f"https://osdr.nasa.gov/osdr/data/osd/files/{osd_id}/?all_files=true&page_size=1000"
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

# Check software versions for reference genome
f767 = get_files(767)
files767 = f767["studies"]["OSD-767"]["study_files"]
sw_file = [f for f in files767 if "software_versions" in f["file_name"].lower()]
if sw_file:
    url = "https://osdr.nasa.gov" + sw_file[0]["remote_url"]
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        print(r.read().decode())


| Program              | Version   | Relevant Links                                                        |
|:---------------------|:----------|:----------------------------------------------------------------------|
| NF_RCP               | 2.1.0     | https://github.com/nasa/GeneLab_Data_Processing/tree/master/RNAseq    |
| Nextflow             | 25.10.3   | https://github.com/nextflow-io/nextflow                               |
| dp_tools             | 1.3.8     | https://github.com/J-81/dp_tools                                      |
| FastQC               | 0.12.1    | https://www.bioinformatics.babraham.ac.uk/projects/fastqc/            |
| MultiQC              | 1.26      | https://multiqc.info/                                                 |
| Cutadapt             | 4.2       | https://cutadapt.readthedocs.io/en/stable/                            |
| TrimGalore!          | 0.6.10    | https://www.bioinformatics.babraham.ac.uk/projects/trim_galore/       |
| STAR             

Extract reference genome from processing info

In [19]:

import urllib.request, json, re
from collections import Counter

def get_files(osd_id):
    url = f"https://osdr.nasa.gov/osdr/data/osd/files/{osd_id}/?all_files=true&page_size=1000"
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

# Download the processing info zip to find reference genome
f767 = get_files(767)
files767 = f767["studies"]["OSD-767"]["study_files"]
proc = [f for f in files767 if "processing_info" in f["file_name"].lower()]
print("Processing info:", proc)
if proc:
    url = "https://osdr.nasa.gov" + proc[0]["remote_url"]
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        import zipfile, io
        z = zipfile.ZipFile(io.BytesIO(r.read()))
        for name in z.namelist():
            print(f"  {name}")
            if name.endswith(('.yml','.yaml','.json','.txt','.md')):
                content = z.read(name).decode(errors='replace')
                # Search for reference genome info
                for line in content.split('\n'):
                    if any(kw in line.lower() for kw in ['reference','genome','fasta','gtf','annotation','organism','itagger','solgenomics','ensembl']):
                        print(f"    >> {line.strip()[:200]}")


Processing info: [{'category': 'GeneLab Processed RNA-Seq Files', 'date_created': 1781824028.088, 'date_updated': 1781824028.088, 'file_name': 'GLDS-709_rna_seq_processing_info_GLbulkRNAseq.zip', 'file_size': 15341, 'organization': 'genelab', 'remote_url': '/geode-py/ws/studies/OSD-767/download?source=datamanager&file=GLDS-709_rna_seq_processing_info_GLbulkRNAseq.zip', 'restricted': False, 'subcategory': 'Processing Info', 'subdirectory': '', 'visible': True}]
  nextflow_run_command_GLbulkRNAseq.txt
    >> -params.reference_source 'ensembl_plants'
    >> -params.reference_version '63'
    >> -params.reference_fasta 'https://ftp.ensemblgenomes.ebi.ac.uk/pub/plants/release-63/fasta/solanum_lycopersicum_gca000188115v5cm/dna/Solanum_lycopersicum_gca000188115v5cm.SL4.0.dna.toplevel.fa.gz'
    >> -params.reference_gtf 'https://ftp.ensemblgenomes.ebi.ac.uk/pub/plants/release-63/gtf/solanum_lycopersicum_gca000188115v5cm/Solanum_lycopersicum_gca000188115v5cm.SL4.0.63.gtf.gz'
  nextflow_processi

Estimate total download sizes

In [21]:

import urllib.request, json

def get_files(osd_id):
    url = f"https://osdr.nasa.gov/osdr/data/osd/files/{osd_id}/?all_files=true&page_size=1000"
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

# Total microbiome FASTQ size
f766 = get_files(766)
files = f766["studies"]["OSD-766"]["study_files"]
fastq = [f for f in files if f["file_name"].endswith(".fastq.gz")]
total_mb = sum(f["file_size"] for f in fastq) / 1e6
print(f"Microbiome FASTQ: {len(fastq)} files, total {total_mb:.1f} MB ({total_mb/1024:.2f} GB)")

# By amplicon
s16 = [f for f in fastq if "_16S-" in f["file_name"]]
its = [f for f in fastq if "_ITS-" in f["file_name"]]
print(f"  16S: {len(s16)} files, {sum(f['file_size'] for f in s16)/1e6:.1f} MB")
print(f"  ITS: {len(its)} files, {sum(f['file_size'] for f in its)/1e6:.1f} MB")

# RNA-seq processed files we need
f767 = get_files(767)
files767 = f767["studies"]["OSD-767"]["study_files"]
need = [f for f in files767 if any(k in f["file_name"] for k in 
    ["RSEM_Unnormalized_Counts", "ISA.zip", "runsheet", "qc_metrics", "software_versions"])]
print(f"\nRNA-seq files to download: {len(need)}")
for f in need:
    print(f"  {f['file_name']}: {f['file_size']/1e6:.2f} MB")
print(f"  Total: {sum(f['file_size'] for f in need)/1e6:.2f} MB")


Microbiome FASTQ: 566 files, total 1346.8 MB (1.32 GB)
  16S: 284 files, 1002.2 MB
  ITS: 282 files, 344.6 MB

RNA-seq files to download: 5
  OSD-767_metadata_OSD-767-ISA.zip: 0.17 MB
  GLDS-709_rna_seq_software_versions_GLbulkRNAseq.md: 0.00 MB
  GLDS-709_rna_seq_bulkRNASeq_v2_runsheet.csv: 0.02 MB
  GLDS-709_rna_seq_RSEM_Unnormalized_Counts_GLbulkRNAseq.csv: 4.47 MB
  GLDS-709_rna_seq_qc_metrics_GLbulkRNAseq.csv: 0.04 MB
  Total: 4.71 MB
